# 📄 NeSy-DocAI: Neuro-Symbolic Document AI Research Notebook

**NeSy-DocAI** là hệ thống bóc tách, tự động sửa lỗi và kiểm toán dữ liệu hóa đơn/chứng từ tài chính dựa trên kiến trúc **Neuro-Symbolic AI (System 1 + System 2)**:
- **System 1 (Visual Perception)**: Trích xuất bố cục ảnh/PDF, Bounding Box 2D $(x_0, y_0, x_1, y_1)$ và Candidate Lattice.
- **System 2 (Symbolic Reasoning)**: Sử dụng **Z3 SMT Solver** kiểm chứng các quy luật đại số kế toán và tự động sửa các lỗi OCR (`O` -> `0`, `l` -> `1`, `S` -> `5`).

In [ ]:
import json
from PIL import Image

from nesy_docai import (
    VisionPerceptionEngine,
    SymbolicSolverEngine,
    TaxMasterDataVerifier,
    AuditExcelExporter,
    BoundingBoxVisualizer,
    PDFDocumentProcessor
)

print("✅ NeSy-DocAI Framework Loaded Successfully!")

## 👁️ Step 1: System 1 Visual Perception (Raw Candidate Extractions)
Dữ liệu OCR thô thường bị nhầm lẫn chữ số (ví dụ: `'1O000'` thay vì `10000`, `'95OO'` thay vì `9500`).

In [ ]:
vision = VisionPerceptionEngine()
raw_data = vision.process_invoice_image("sample_invoice.png")

print("📥 Raw OCR Extractions (Contains OCR noise '1O000' and '95OO'):")
print(json.dumps(raw_data, ensure_ascii=False, indent=2))

## 🧠 Step 2: System 2 Symbolic Reasoning (Z3 Presburger SMT Solver)
Z3 Solver thực thi các ràng buộc kế toán:
- Thành tiền = Số lượng x Đơn giá
- Subtotal = Tổng các Thành tiền
- Total = Subtotal + Tax

Tự bù lỗi số học trong **~2.8ms** và cấp chứng nhận `SAT`.

In [ ]:
solver = SymbolicSolverEngine(vision_engine=vision)
verified_record = solver.solve_and_verify(raw_data)

print(f"⚙️ SMT Status: {verified_record['audit_status']}")
print("✨ Corrected & Verified Financial Record:")
print(json.dumps(verified_record, ensure_ascii=False, indent=2))

## 🏛️ Step 3: Tax Master Data & Enterprise Legal Verification

In [ ]:
tax_verifier = TaxMasterDataVerifier()
tax_info = tax_verifier.verify_tax_id(verified_record['seller_tax_id'])
verified_record['tax_verification'] = tax_info

print("🏛️ General Department of Taxation Verification:")
print(json.dumps(tax_info, ensure_ascii=False, indent=2))

## 🖼️ Step 4: Pixel-Level Provenance & Bounding Box Annotation

In [ ]:
visualizer = BoundingBoxVisualizer()
pdf_proc = PDFDocumentProcessor()

# Render invoice page with bounding box annotations
invoice_img = pdf_proc.render_dummy_pdf_page_image(page_num=1)
annotated_img = visualizer.annotate_invoice(invoice_img, verified_record)

# Display annotated image inline
display(annotated_img)

## 📊 Step 5: Export Multi-Sheet Excel Audit Report

In [ ]:
exporter = AuditExcelExporter()
excel_path = exporter.export_to_excel([verified_record], output_filepath="invoice_audit_report.xlsx")

print(f"💾 Excel Audit Report saved to: {excel_path}")